# RANDE NY — Sample Notebook

Deploy and call the **RANDE NY** race/ethnicity imputation model package from AWS Marketplace in **your own AWS account**. Your input data never leaves your account.

**Scope:** New York State. **Output:** privacy-preserving aggregate race/ethnicity summaries by census tract and ZCTA (White/Black/Hispanic/Asian + an "unclassified" residual) by default; per-row probability weights on request.

> Outputs are **statistical estimates** of the race/ethnicity signaled by name and geography — **not** statements of any individual's actual or self-identified race. Do not use for decisions about specific individuals. See the product EULA.

## 1. Setup
After subscribing on AWS Marketplace, copy the **model package ARN** for your Region into `MODEL_PACKAGE_ARN` below.

In [ ]:
import sagemaker, boto3
from sagemaker import ModelPackage, get_execution_role

sess = sagemaker.Session()
role = get_execution_role()
region = sess.boto_region_name

# Paste the model package ARN from your AWS Marketplace subscription:
MODEL_PACKAGE_ARN = "arn:aws:sagemaker:<region>:<your-acct>:model-package/<rande-ny-...>"
print(region, role)

## 2. Sample input (CSV)
Header: `fname,mname,lname,housenumber,street,city,state,zip` (New York addresses).

In [ ]:
sample_csv = '''fname,mname,lname,housenumber,street,city,state,zip
Maria,,Garcia,55,Broadway,New York,NY,10006
Wei,,Chen,28,Mott St,New York,NY,10013
Aisha,M,Johnson,742,Grand Ave,Brooklyn,NY,11238
Robert,L,Williams,15,Elm St,Buffalo,NY,14201
'''
open('sample.csv','w').write(sample_csv)
print(sample_csv)

## 3. Real-time endpoint

In [ ]:
model = ModelPackage(role=role, model_package_arn=MODEL_PACKAGE_ARN, sagemaker_session=sess)
predictor = model.deploy(initial_instance_count=1, instance_type='ml.m5.xlarge', endpoint_name='rande-ny-demo')
print('endpoint InService')

In [ ]:
rt = boto3.client('sagemaker-runtime', region_name=region)

# Default = aggregate tract/ZCTA summaries
agg = rt.invoke_endpoint(EndpointName='rande-ny-demo', ContentType='text/csv', Body=sample_csv)
print(agg['Body'].read().decode())

# Per-row predictions + weights
rows = rt.invoke_endpoint(EndpointName='rande-ny-demo', ContentType='text/csv', Body=sample_csv,
                          CustomAttributes='mode=rows')
print(rows['Body'].read().decode())

## 4. Tear down (stop the hourly charge)

In [ ]:
predictor.delete_endpoint(delete_endpoint_config=True)

## 5. Batch Transform (large files)
For whole-file aggregate output use `split_type='None'`. Upload your CSV to S3 first.
```python
t = model.transformer(instance_count=1, instance_type='ml.m5.xlarge', accept='text/csv',
                      output_path='s3://YOUR-BUCKET/rande-out/')
t.transform('s3://YOUR-BUCKET/input.csv', content_type='text/csv', split_type='None')
t.wait()
```